In [ ]:
from datasets.packaged_modules.pandas.pandas import Pandas
%load_ext autoreload
%autoreload 2
import os
from dotenv import load_dotenv
from TextMiningBasedSatdDetectorModel import TextMiningBasedSatdDetectorModel
from SimpleOutputLabelConverter import SimpleOutputLabelConverter
from constant import *


In [ ]:
load_dotenv()
simple_output_label_converter = SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)
BASE_SATD_DETECTOR_DIRECTORY = os.getenv('BASE_SATD_DETECTOR_DIRECTORY')

# Training Dataset Preparation

In [ ]:
def init_liu_detector_directory(variant_name:str, train_df:pd.DataFrame, test_df:pd.DataFrame):
  variant_directory = f"{BASE_SATD_DETECTOR_DIRECTORY}/models/{variant_name}"
  os.makedirs(variant_directory, exist_ok=True)
  os.makedirs(os.path.join(variant_directory, 'models'), exist_ok=True)
  train_df['text'].str.replace('\n', '\t').to_csv(f'{variant_directory}/comments.txt', index=False, header=False)
  train_df['label'].str.lower().map({'yes': 'Yes', 'no': 'No'}).to_csv(f'{variant_directory}/labels.txt', index=False, header=False)
  train_df.assign(project='train')['project'].to_csv(f'{variant_directory}/projects.txt', index=False, header=False)
  return variant_directory


In [ ]:
pretrained_satd_detector = TextMiningBasedSatdDetectorModel('detect', 'pretrained-liu-detector', simple_output_label_converter, init_liu_detector_directory("default", detect_train_df, detect_test_df))
pretrained_satd_detector.fit(detect_train_dataset)
pretrained_satd_detector.predict(detect_test_dataset, DATASET_NAME)

In [ ]:
trained_detector = TextMiningBasedSatdDetectorModel('detect', 'trained-liu-detector',simple_output_label_converter, init_liu_detector_directory("default", detect_train_df, detect_test_df), retrain=True)
trained_detector.fit(detect_train_dataset)
trained_detector.predict(detect_test_dataset, DATASET_NAME)

5-Fold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd

df = pd.concat([detect_train_df, detect_test_df])

X = df["text"]
y = df["label"]
groups = df["repository"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]
    fold_suffix = f"5fcv-{fold + 1}"
    print(f'Fold {fold_suffix}: {len(train_df)} train and {len(test_df)} test samples')
    trained_detector = TextMiningBasedSatdDetectorModel('detect', f'trained-liu-detector-{fold_suffix}',
                                                        simple_output_label_converter,
                                                        init_liu_detector_directory(fold_suffix, train_df, test_df),
                                                        retrain=True)
    trained_detector.fit(Dataset.from_pandas(train_df))
    trained_detector.predict(Dataset.from_pandas(test_df), DATASET_NAME)
    break
